Imports


In [60]:
#imports 
import sqlite3
import pandas as pd
from datetime import datetime
import os 


#verbinden aan databases
DWH = sqlite3.connect("../../Database/DWH 3.db")
SDM = sqlite3.connect("../../Database/SDM2.db")

#huidige datum, nodig voor SDC2
today = datetime.now().strftime("%Y-%m-%d")

#log data tijdig opslaan
logbook = []


def log(table, entity_id, action, status, message=""):
    logbook.append({
        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "table": table,
        "entity_id": entity_id,
        "action": action,
        "status": status,
        "message": message
    })


def export_log():
    df_log = pd.DataFrame(logbook)
    df_log.to_excel("DWH_logboek1.xlsx", index=False)
    print("Logboek exported to DWH_logboek.xlsx")



In [61]:
#data inladen uit sdm
def load(table):
    try:
        #haalt alles uit sdm
        return pd.read_sql_query(f"SELECT * FROM {table}", SDM)
    #als fout 
    except Exception as e:
        #log fout => tabel niet bestaat
        log(table, None, "LOAD", "FAIL", str(e))
        return pd.DataFrame()

#Zet tijd om naar datetime object
def parse_time(t):
    
    return datetime.strptime(str(t).split(".")[0], "%H:%M:%S")

In [62]:
#sk halen uit 
def get_sk(table, key, value):
    # Mapping tussen tabel en surrogate key kolom
    mapping = {
        "Dim_Klant": "klant_sk",
        "Dim_Fiets": "fiets_sk",
        "Dim_Monteur": "monteur_sk",
        "Dim_Datum": "datum_sk",
        "Dim_Accessoire": "accessoire_sk"
    }


    sk_col = mapping[table]

    query = f"SELECT {sk_col} FROM {table} WHERE {key}=?"

    # Voor SCD2 tabellen => alleen actuele record gebruiken
    if table in ["Dim_Klant", "Dim_Fiets"]:
        query += " AND is_actueel=1"

    df = pd.read_sql_query(query, DWH, params=(value,))
    return None if df.empty else int(df.iloc[0, 0])

In [63]:
# SDC 2 voorbeeld 
#Combineer klantdata uit meerdere bronnen
df_klant = pd.concat([
    load("Accessoireverkoop_Klant"),
    load("Fietsverkoop_Klant")
], ignore_index=True).drop_duplicates("klantnr")


# Loop door elke klant row
for _, r in df_klant.iterrows():
    try:

        # Check of klant al bestaat
        existing = pd.read_sql_query("""
            SELECT * FROM Dim_Klant
            WHERE klantnr=? AND is_actueel=1
        """, DWH, params=(r["klantnr"],))

        # Leeftijd berekenen => afgeleide waarden 
        geboortejaar = int(str(r["geboortedatum"])[:4])
        leeftijd = 2026 - geboortejaar

        # Leeftijdsgroep bepalen => afgeleide waarden
        groep = (
            "0-17" if leeftijd < 18 else
            "18-29" if leeftijd < 30 else
            "30-49" if leeftijd < 50 else "50+"
        )

         # ALS klant nog niet bestaat => Insert
        if existing.empty:
            # Insert
            DWH.execute("""
                INSERT INTO Dim_Klant (
                    klantnr, naam, woonplaats, geslacht, geboortedatum,
                    leeftijdsgroep, geldig_vanaf, geldig_tot, is_actueel
                )
                VALUES (?, ?, ?, ?, ?, ?, ?, NULL, 1)
            """, (
                r["klantnr"], r["naam"], r["woonplaats"],
                r["geslacht"], r["geboortedatum"],
                groep, today
            ))
            #sucess log
            log("Dim_Klant", r["klantnr"], "INSERT", "SUCCESS")

        # Als hij al bestaat update
        else:
            #haal de bestaande klant uit de database op als DataFrame.
            #pak de eerste en enige rij
            old = existing.iloc[0]

             # check of data veranderd is
            if (
                old["naam"] != r["naam"] or
                old["woonplaats"] != r["woonplaats"] or
                old["geslacht"] != r["geslacht"]
            ):
                
                 # Sluit oude record (SCD2)
                DWH.execute("""
                    UPDATE Dim_Klant
                    SET geldig_tot=?, is_actueel=0
                    WHERE klant_sk=?
                """, (today, old["klant_sk"]))

                 # Voeg nieuwe versie toe
                DWH.execute("""
                    INSERT INTO Dim_Klant (
                        klantnr, naam, woonplaats, geslacht, geboortedatum,
                        leeftijdsgroep, geldig_vanaf, geldig_tot, is_actueel
                    )
                    VALUES (?, ?, ?, ?, ?, ?, ?, NULL, 1)
                """, (
                    r["klantnr"], r["naam"], r["woonplaats"],
                    r["geslacht"], r["geboortedatum"],
                    groep, today
                ))

                log("Dim_Klant", r["klantnr"], "UPDATE_SCD2", "SUCCESS")
             # Geen verandering → skip
            else:
                log("Dim_Klant", r["klantnr"], "SKIP", "NO CHANGE")

    except Exception as e:
        log("Dim_Klant", r.get("klantnr"), "ERROR", "FAIL", str(e))

#commit
DWH.commit()


In [64]:
#SCD1 + upsert 
#Combineer data uit meerdere bronnen => complete dataset van sdm over monteur 
df_monteur = pd.concat([
    load("Accessoireverkoop_Monteur"),
    load("Fietsverkoop_Monteur"),
    load("Onderhoud_Monteur")
], ignore_index=True).drop_duplicates("monteurnr")

# upsert => insert of update
#als
# als monteurnr al bestaat in de tabel => On Conflict => update => 
for _, r in df_monteur.iterrows():
    try:
        DWH.execute("""
            INSERT INTO Dim_Monteur (
                monteurnr, naam, uurloon,
                filiaal_naam, filiaal_provincie
            )
            VALUES (?, ?, ?, ?, ?)
            ON CONFLICT(monteurnr)
            DO UPDATE SET
                naam=excluded.naam,
                uurloon=excluded.uurloon
        """, (
            r["monteurnr"], r["naam"], r["uurloon"],
            "unknown", "unknown"
        ))

        #gelukt
        log("Dim_Monteur", r["monteurnr"], "UPSERT", "SUCCESS")

    #
    except Exception as e:
        log("Dim_Monteur", r.get("monteurnr"), "ERROR", "FAIL", str(e))

DWH.commit()


In [65]:
#SCD 2 => elke verandering nieuwe rij
#
#combineer data 
df_fiets = pd.concat([
    load("Fietsverkoop_Fiets"),
    load("Onderhoud_Fiets"),
    load("Fiets_Inkoop_Fiets")
], ignore_index=True).drop_duplicates("fietsnr")

#loop door elke rij
for _, r in df_fiets.iterrows():
    try:
        #afgeleid meetwaarden
        prijsklasse = (
            "budget" if r["standaardprijs"] < 500 else
            "mid" if r["standaardprijs"] < 1500 else "premium"
        )

        # We kijken of deze fiets al bestaat als actuele versie
        existing = pd.read_sql_query("""
            SELECT * FROM Dim_Fiets
            WHERE fietsnr=? AND is_actueel=1
        """, DWH, params=(r["fietsnr"],))

         # Als nog niet bestaat => insert
        if existing.empty:
            DWH.execute("""
                INSERT INTO Dim_Fiets (
                    fietsnr, soort, merk, type, kleur,
                    standaardprijs, inkoopprijs, prijsklasse,
                    fabrikant_naam, fabrikant_plaats,
                    geldig_vanaf, geldig_tot, is_actueel
                )
                VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, NULL, 1)
            """, (
                r["fietsnr"], r["soort"], r["merk"], r["type"],
                r["kleur"], r["standaardprijs"], r["inkoopprijs"],
                prijsklasse, "UNKNOWN", "UNKNOWN", today
            ))

            #success log
            log("Dim_Fiets", r["fietsnr"], "INSERT", "SUCCESS")

        #als fiets al bestaat
        else:
            #ophalen
            old = existing.iloc[0]

            #checken of hij dezelfde is 
            if (
                old["kleur"] != r["kleur"] or
                old["standaardprijs"] != r["standaardprijs"]
            ):
                #oude versie sluiten => actueel = 0, geldig = today
                DWH.execute("""
                    UPDATE Dim_Fiets
                    SET geldig_tot=?, is_actueel=0
                    WHERE fiets_sk=?
                """, (today, old["fiets_sk"]))

                #insert nieuwe
                DWH.execute("""
                    INSERT INTO Dim_Fiets (
                        fietsnr, soort, merk, type, kleur,
                        standaardprijs, inkoopprijs, prijsklasse,
                        fabrikant_naam, fabrikant_plaats,
                        geldig_vanaf, geldig_tot, is_actueel
                    )
                    VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, NULL, 1)
                """, (
                    r["fietsnr"], r["soort"], r["merk"], r["type"],
                    r["kleur"], r["standaardprijs"], r["inkoopprijs"],
                    prijsklasse, "UNKNOWN", "UNKNOWN", today
                ))

                #sucess update log
                log("Dim_Fiets", r["fietsnr"], "UPDATE_SCD2", "SUCCESS")
            else:
                # geen verandering
                log("Dim_Fiets", r["fietsnr"], "SKIP", "NO CHANGE")

    
    except Exception as e:
        log("Dim_Fiets", r.get("fietsnr"), "ERROR", "FAIL", str(e))

DWH.commit()


In [66]:
#SCD1 + upsert voorbeeld
#geen history want data word overgeschreven 
#data combineeren => volledig dataset 
df_acc = pd.concat([
    load("Accessoireverkoop_Accessoire"),
    load("Accessoire_Inkoop_Accessoire")
], ignore_index=True).drop_duplicates("accessoirenr")

#loop door elke rij
for _, r in df_acc.iterrows():
    try:
        #insert rij 
        #on conflict met key => rij bestaat al => switch naar update
        #r => data van sdm
        DWH.execute("""
            INSERT INTO Dim_Accessoire (
                accessoirenr, naam, soort,
                standaardprijs, inkoopprijs,
                leverancier_naam, leverancier_woonplaats
            )
            VALUES (?, ?, ?, ?, ?, ?, ?)
            ON CONFLICT(accessoirenr)
            DO UPDATE SET
                naam=excluded.naam,
                soort=excluded.soort,
                standaardprijs=excluded.standaardprijs,
                inkoopprijs=excluded.inkoopprijs
        """, (
            r["accessoirenr"], r["naam"], r["soort"],
            r["standaardprijs"], r["inkoopprijs"],
            "UNKNOWN", "UNKNOWN"
        ))

        #success log
        log("Dim_Accessoire", r["accessoirenr"], "UPSERT", "SUCCESS")

    except Exception as e:
        log("Dim_Accessoire", r.get("accessoirenr"), "ERROR", "FAIL", str(e))

DWH.commit()


In [67]:
#SCD1 logica => alleen insert
#combineer data
df_dates = pd.concat([
    load("Fietsverkoop_Fiets_Verkoop")[["datum"]],
    load("Accessoireverkoop_Accessoire_Verkoop")[["datum"]],
    load("Onderhoud_Onderhoud")[["datum"]]
], ignore_index=True).drop_duplicates()

# We lopen door elke rij heen
#bestaat al? als hij nog niet bestaat dan insert 
for d in df_dates["datum"]:
    try:
        exists = pd.read_sql_query(
            "SELECT 1 FROM Dim_Datum WHERE datum=?",
            DWH,
            params=(d,)
        )

        if exists.empty:
            #afgeleid
            dt = datetime.strptime(d, "%Y-%m-%d")
            kwartaal = (dt.month - 1)//3 + 1

            DWH.execute("""
                INSERT INTO Dim_Datum (datum, dag, maand, jaar, kwartaal)
                VALUES (?, ?, ?, ?, ?)
            """, (d, dt.day, dt.month, dt.year, kwartaal))

            log("Dim_Datum", d, "INSERT", "SUCCESS")

    except Exception as e:
        log("Dim_Datum", d, "ERROR", "FAIL", str(e))

DWH.commit()

In [68]:
#feit loading proces => etl stap 
#We laden de verkoopdata van accessoires uit de SDM
df = load("Accessoireverkoop_Accessoire_Verkoop")

# We lopen door elke regel
for _, r in df.iterrows():
    try:
        #get sk van dim 
        # vertalen bk (klantnr, accessoirenr, etc.) =>  naar surrogate keys (SK) uit de DWH dim
        klant_sk = get_sk("Dim_Klant", "klantnr", r["klant"])
        acc_sk = get_sk("Dim_Accessoire", "accessoirenr", r["accessoire"])
        monteur_sk = get_sk("Dim_Monteur", "monteurnr", r["monteur"])
        datum_sk = get_sk("Dim_Datum", "datum", r["datum"])

        # 
        if None in (klant_sk, acc_sk, monteur_sk, datum_sk):
            log("Fact_Accessoire_Verkoop", r["accessoire_verkoopnr"], "SKIP", "MISSING SK")
            continue

        #afgeleid
        total = r["aantal"] * r["verkoopprijs"]


        #insert naar feit tabel
        # met verwijzingen naar dim => sk
        DWH.execute("""
            INSERT INTO Fact_Accessoire_Verkoop (
                accessoire_verkoop, datum_sk, klant_sk,
                accessoire_sk, monteur_sk,
                aantal, verkoopprijs, totale_prijs
            )
            VALUES (?, ?, ?, ?, ?, ?, ?, ?)
        """, (
            r["accessoire_verkoopnr"],
            datum_sk, klant_sk, acc_sk, monteur_sk,
            r["aantal"], r["verkoopprijs"], total
        ))

        log("Fact_Accessoire_Verkoop", r["accessoire_verkoopnr"], "INSERT", "SUCCESS")

    except Exception as e:
        log("Fact_Accessoire_Verkoop", r.get("accessoire_verkoopnr"), "ERROR", "FAIL", str(e))

DWH.commit()

In [ ]:
#feit => ETL proces
# We laden de onderhoudsdata uit de SDM
df = load("Onderhoud_Onderhoud")

#loop door elke rij
for _, r in df.iterrows():
    try:
         # Haal sk op
        datum_sk = get_sk("Dim_Datum", "datum", r["datum"])
        fiets_sk = get_sk("Dim_Fiets", "fietsnr", r["fiets"])
        monteur_sk = get_sk("Dim_Monteur", "monteurnr", r["monteur"])

        if None in (datum_sk, fiets_sk, monteur_sk):
            log("Fact_Onderhoud", r.get("onderhoudnr"), "SKIP", "MISSING SK")
            continue

        #tijd omzetten => voorbereiding van duur 
        start = parse_time(r["starttijd"])
        end = parse_time(r["eindtijd"])

        # Duur berekenen in minuten => afgeleid 
        duur = int((end - start).total_seconds() / 60)

        # Uurloon ophalen
        uurloon_df = pd.read_sql_query(
            "SELECT uurloon FROM Dim_Monteur WHERE monteur_sk=?",
            DWH,
            params=(monteur_sk,)
        )

        # Kosten berekenen
        if uurloon_df.empty:
            log("Fact_Onderhoud", r.get("onderhoudnr"), "SKIP", "NO UURLOON")
            continue


        uurloon = uurloon_df.iloc[0, 0]

        # Kosten berekenen
        kosten = (duur / 60) * uurloon

         # Insert 
        DWH.execute("""
            INSERT INTO Fact_Onderhoud (
                onderhoud, datum_sk, fiets_sk, monteur_sk,
                duur_minuten, arbeidskosten
            )
            VALUES (?, ?, ?, ?, ?, ?)
        """, (
            r["onderhoudnr"],
            datum_sk,
            fiets_sk,
            monteur_sk,
            duur,
            kosten
        ))

        log("Fact_Onderhoud", r.get("onderhoudnr"), "INSERT", "SUCCESS")

    except Exception as e:
        log("Fact_Onderhoud", r.get("onderhoudnr"), "ERROR", "FAIL", str(e))

DWH.commit()

print("Fact_Onderhoud ingeladen")


Fact_Onderhoud ingeladen


In [70]:
export_log()


Logboek exported to DWH_logboek.xlsx


In [71]:
tables = pd.read_sql_query(
    "SELECT name FROM sqlite_master WHERE type='table'",
    DWH
)

print(tables)


                      name
0                Dim_Klant
1          sqlite_sequence
2           Dim_Accessoire
3              Dim_Monteur
4                Dim_Datum
5                Dim_Fiets
6       Fact_Fiets_Verkoop
7           Fact_Onderhoud
8  Fact_Accessoire_Verkoop


In [72]:
print(pd.read_sql_query("SELECT COUNT(*) FROM Dim_Klant", DWH))
print(pd.read_sql_query("SELECT COUNT(*) FROM Dim_Fiets", DWH))
print(pd.read_sql_query("SELECT COUNT(*) FROM Fact_Onderhoud", DWH))



   COUNT(*)
0        25
   COUNT(*)
0        75
   COUNT(*)
0       400
